# QLoRA NF4 Retrain + Baseline + GGUF Export (T4)

Retrains `qwen3-4b-instruct-2507` with actual QLoRA NF4 4-bit quantization on Colab T4,
runs baseline eval with the SAME model (no adapter), then exports GGUF Q8_0.

**Upload required:**
- `train.jsonl` from `data/splits/recovered-balanced/train.jsonl`
- `val.jsonl` from `data/splits/recovered-balanced/val.jsonl`

**Output (download all from /content/):**
- `baseline-eval-qwen3-4b.json` — baseline (zero-shot, same model)
- `eval-results-qlora.json` — fine-tuned eval
- `training-summary.json` — proves `quantization_mode: 4bit-qlora`
- `adapter-qlora-final.zip` — the QLoRA adapter
- GGUF file — merged Q8_0 artifact

**Time:** ~1h baseline eval + ~2-4h training + ~10 min GGUF on T4

In [ ]:
%%capture
%pip install -q peft>=0.12.0 transformers>=4.46.0 accelerate>=1.0.0 bitsandbytes>=0.44.0 datasets>=3.0.0

In [ ]:
import subprocess, os
from pathlib import Path

REPO_URL = "https://github.com/wikiepeidia/Internship-project.git"
REPO_ROOT = Path("/content/vnphish-repo")
TRAIN_SPLIT = Path("/content/train.jsonl")
VAL_SPLIT = Path("/content/val.jsonl")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull"], check=False)

for f in [TRAIN_SPLIT, VAL_SPLIT]:
    if not f.exists():
        raise FileNotFoundError(f"Upload {f.name} to /content/ first!")

print(f"Train: {TRAIN_SPLIT} ({TRAIN_SPLIT.stat().st_size // 1024} KB)")
print(f"Val: {VAL_SPLIT} ({VAL_SPLIT.stat().st_size // 1024} KB)")

In [ ]:
import json, os
from pathlib import Path

MODEL_ROOT = Path("/content/model-artifacts")
REGISTRY_PATH = MODEL_ROOT / "manifests" / "model-registry.json"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
REGISTRY_PATH.parent.mkdir(parents=True, exist_ok=True)
VERSION_TAG = "qlora-final-2026-06"

registry = {
    "version_tag": VERSION_TAG,
    "selection": {
        "baseline_winner_id": "qwen3-4b-instruct-2507",
        "runner_up_id": "qwen3.5-4b",
        "selection_notes": "QLoRA NF4 retrain on T4"
    },
    "scorecards": [],
    "artifacts": []
}
REGISTRY_PATH.write_text(json.dumps(registry, indent=2), encoding="utf-8")

os.environ["MODEL_ARTIFACT_ROOT"] = str(MODEL_ROOT)
os.environ["MODEL_REGISTRY_PATH"] = str(REGISTRY_PATH)
print("Config ready.")

In [ ]:
from huggingface_hub import snapshot_download

BASE_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
BASE_LOCAL = MODEL_ROOT / "base" / "qwen3-4b-instruct-2507"
BASE_LOCAL.mkdir(parents=True, exist_ok=True)

print(f"Downloading {BASE_MODEL_ID}...")
snapshot_download(
    repo_id=BASE_MODEL_ID, local_dir=str(BASE_LOCAL),
    ignore_patterns=["*.msgpack", "flax_model*", "tf_model*"]
)
print(f"Done. Size: {sum(f.stat().st_size for f in BASE_LOCAL.rglob('*') if f.is_file()) // (1024*1024)} MB")

## Step 4a: Baseline eval (SAME model, NO adapter, zero-shot)

Run this BEFORE training so baseline and fine-tuned use the EXACT same base model.
This fixes the previous baseline which incorrectly used Qwen3.5-4B.

In [ ]:
import json, re, torch
from pathlib import Path
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer

LABELS = ["bank_impersonation", "zalo_social_engineering", "task_scam", "benign"]

LABEL_ALIASES = {
    "bank_impersonation": "bank_impersonation", "bank impersonation": "bank_impersonation",
    "zalo_social_engineering": "zalo_social_engineering", "zalo social engineering": "zalo_social_engineering",
    "social_engineering": "zalo_social_engineering", "social engineering": "zalo_social_engineering",
    "task_scam": "task_scam", "task scam": "task_scam",
    "benign": "benign", "safe": "benign", "legitimate": "benign", "not phishing": "benign",
}

BASELINE_PROMPT = """You are a Vietnamese financial phishing detector.
Classify the following message into exactly one category:
- bank_impersonation (fake bank messages asking for OTP/credentials)
- zalo_social_engineering (social manipulation via Zalo/chat platforms)
- task_scam (fake job/task offers demanding payment)
- benign (legitimate, safe message)

Reply with ONLY the category name. Do not explain."""

def extract_label(raw):
    text = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip().lower()
    for lbl in LABELS:
        if lbl in text: return lbl
    for alias, canonical in sorted(LABEL_ALIASES.items(), key=lambda x: -len(x[0])):
        if alias in text: return canonical
    if any(w in text for w in ["bank", "ngân hàng", "otp"]): return "bank_impersonation"
    if any(w in text for w in ["zalo", "social"]): return "zalo_social_engineering"
    if any(w in text for w in ["task", "việc", "job"]): return "task_scam"
    if any(w in text for w in ["safe", "legit", "normal"]): return "benign"
    return "unknown"

print(f"Loading base model for BASELINE eval (no adapter)...")
base_model = AutoModelForCausalLM.from_pretrained(str(BASE_LOCAL), torch_dtype=torch.float16, device_map="auto")
base_tokenizer = AutoTokenizer.from_pretrained(str(BASE_LOCAL))
base_model.eval()
print(f"Loaded on {torch.cuda.get_device_name(0)}")

val_rows = [json.loads(l) for l in open(VAL_SPLIT, encoding="utf-8")]
print(f"\nRunning BASELINE (zero-shot, no adapter) on {len(val_rows)} val rows...")

baseline_results = []
for i, row in enumerate(val_rows):
    messages = [{"role": "user", "content": BASELINE_PROMPT + "\n\nMessage: " + row["text"]}]
    try:
        prompt = base_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        prompt = base_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        out = base_model.generate(**inputs, max_new_tokens=64, do_sample=False, pad_token_id=base_tokenizer.eos_token_id)
    raw = base_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    pred = extract_label(raw)
    baseline_results.append({"true": row["label"], "pred": pred})
    if (i+1) % 25 == 0:
        acc = sum(1 for r in baseline_results if r["true"]==r["pred"]) / len(baseline_results)
        unk = sum(1 for r in baseline_results if r["pred"]=="unknown")
        print(f"  {i+1}/{len(val_rows)} — accuracy: {acc:.3f}, unknowns: {unk}")

# Compute baseline metrics
tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
for r in baseline_results:
    if r["true"] == r["pred"]: tp[r["true"]] += 1
    else: fp[r["pred"]] += 1; fn[r["true"]] += 1

print("\n" + "="*60)
print("BASELINE (Qwen3-4B-Instruct-2507, zero-shot, NO adapter)")
print("="*60)
baseline_metrics = {}
for lbl in LABELS:
    sup = sum(1 for r in baseline_results if r["true"]==lbl)
    p = tp[lbl]/(tp[lbl]+fp[lbl]) if (tp[lbl]+fp[lbl])>0 else 0
    rec = tp[lbl]/(tp[lbl]+fn[lbl]) if (tp[lbl]+fn[lbl])>0 else 0
    f1 = 2*p*rec/(p+rec) if (p+rec)>0 else 0
    baseline_metrics[lbl] = {"precision": round(p,4), "recall": round(rec,4), "f1": round(f1,4), "support": sup}
    print(f"  {lbl:35s} P={p:.4f} R={rec:.4f} F1={f1:.4f} n={sup}")

correct = sum(1 for r in baseline_results if r["true"]==r["pred"])
baseline_macro_f1 = sum(m["f1"] for m in baseline_metrics.values()) / 4
print(f"\nBaseline Macro F1: {baseline_macro_f1:.4f}  Accuracy: {correct}/{len(baseline_results)}")

baseline_output = {
    "base_model": "Qwen/Qwen3-4B-Instruct-2507",
    "holdout_size": len(baseline_results),
    "macro_f1": round(baseline_macro_f1, 4),
    "accuracy": round(correct/len(baseline_results), 4),
    "per_class": baseline_metrics,
    "predictions": baseline_results,
}
Path("/content/baseline-eval-qwen3-4b.json").write_text(json.dumps(baseline_output, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nSaved to /content/baseline-eval-qwen3-4b.json")

# Free GPU memory before training
del base_model
torch.cuda.empty_cache()
print("Base model unloaded, GPU memory freed for QLoRA training.")

In [ ]:
import subprocess, sys, os

# NOTE: NO --full-precision flag! This enables actual QLoRA NF4.
train_cmd = [
    sys.executable, "-m", "src.model_adaptation.cli", "train",
    "--candidate", "baseline-winner",
    "--version-tag", VERSION_TAG,
    "--train-split", str(TRAIN_SPLIT),
    "--val-split", str(VAL_SPLIT),
    "--output-root", str(MODEL_ROOT),
    "--registry-path", str(REGISTRY_PATH),
    "--device", "cuda",
    # NO --full-precision → QLoRA NF4 4-bit is used
]

print("Training with QLoRA NF4 (4-bit base + full-precision LoRA adapters)")
print("Command:", " ".join(train_cmd))
print("This may take 2-4 hours on T4...")

result = subprocess.run(train_cmd, cwd=str(REPO_ROOT), env={**os.environ},
                        capture_output=True, text=True)
if result.returncode != 0:
    print("STDOUT:", result.stdout[-2000:])
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError(f"Training failed with exit code {result.returncode}")
print("Training complete!")

# Verify quantization mode
adapter_dir = MODEL_ROOT / VERSION_TAG / "qwen3-4b-instruct-2507" / "adapter"
summary = json.loads((adapter_dir / "training-summary.json").read_text())
print(f"\nQuantization mode: {summary['quantization_mode']}")
print(f"Train examples: {summary['train_examples']}")
print(f"Val examples: {summary['val_examples']}")
assert summary['quantization_mode'] == '4bit-qlora', f"Expected 4bit-qlora but got {summary['quantization_mode']}!"

## Step 5: Convert to GGUF Q8_0

In [ ]:
import subprocess, sys

convert_cmd = [
    sys.executable, "-m", "src.model_adaptation.cli", "convert",
    "--candidate", "baseline-winner",
    "--version-tag", VERSION_TAG,
    "--output-root", str(MODEL_ROOT),
    "--registry-path", str(REGISTRY_PATH),
]

print("Converting merged adapter to GGUF Q8_0...")
result = subprocess.run(convert_cmd, cwd=str(REPO_ROOT), env={**os.environ},
                        capture_output=True, text=True)
if result.returncode != 0:
    print("STDOUT:", result.stdout[-2000:])
    print("STDERR:", result.stderr[-2000:])
    # Fallback: try manual convert
    print("\nFallback: attempting manual GGUF conversion...")
    # Merge adapter first
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch
    
    print("Loading base + adapter for merge...")
    base = AutoModelForCausalLM.from_pretrained(str(BASE_LOCAL), torch_dtype=torch.float16, device_map="cpu")
    merged = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload()
    merge_dir = MODEL_ROOT / VERSION_TAG / "merged"
    merge_dir.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(str(merge_dir))
    AutoTokenizer.from_pretrained(str(BASE_LOCAL)).save_pretrained(str(merge_dir))
    print(f"Merged model saved to {merge_dir}")
    
    # Convert to GGUF
    gguf_out = MODEL_ROOT / VERSION_TAG / "gguf-laptop.gguf"
    convert_script = REPO_ROOT / "scripts" / "convert_hf_to_gguf.py"
    if not convert_script.exists():
        %pip install -q gguf
        convert_script = "convert_hf_to_gguf.py"  # from gguf package
    subprocess.run([sys.executable, str(convert_script), str(merge_dir),
                    "--outfile", str(gguf_out), "--outtype", "q8_0"], check=True)
    print(f"GGUF saved to {gguf_out}")
else:
    print("GGUF conversion complete!")
    gguf_candidates = list(MODEL_ROOT.rglob("*.gguf"))
    for g in gguf_candidates:
        print(f"  {g} ({g.stat().st_size // (1024*1024)} MB)")

## Step 6: Evaluate on val split

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from collections import defaultdict

print("Loading base + QLoRA adapter for eval...")
eval_model = AutoModelForCausalLM.from_pretrained(str(BASE_LOCAL), torch_dtype=torch.float16, device_map="auto")
eval_model = PeftModel.from_pretrained(eval_model, str(adapter_dir))
eval_tokenizer = AutoTokenizer.from_pretrained(str(BASE_LOCAL))
eval_model.eval()

LABELS = ["bank_impersonation", "zalo_social_engineering", "task_scam", "benign"]
_CANDIDATE_ID = "Qwen/Qwen3-4B-Instruct-2507"
_SCHEMA = json.dumps({"label": "bank_impersonation | zalo_social_engineering | task_scam | benign",
    "risk_tier": "benign | suspicious | high-risk",
    "suspicious_spans": ["exact suspicious substrings"],
    "xai_explanation": "localized explanation"}, ensure_ascii=False)

def classify(text):
    instruction = f"Candidate: {_CANDIDATE_ID}\nYou are fine-tuning a local Vietnamese phishing detector.\nAnalyze the following raw message text and produce a structured response.\nResponse schema: {_SCHEMA}\nMessage text: {text}"
    full = f"### Instruction\n{instruction}\n\n### Response\n"
    inputs = eval_tokenizer(full, return_tensors="pt").to(eval_model.device)
    with torch.no_grad():
        out = eval_model.generate(**inputs, max_new_tokens=64, do_sample=False)
    raw = eval_tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip().lower()
    return next((lbl for lbl in LABELS if lbl in raw), "benign")

val_rows = [json.loads(l) for l in open(VAL_SPLIT, encoding="utf-8")]
print(f"Evaluating {len(val_rows)} val rows...")
results = []
for i, row in enumerate(val_rows):
    pred = classify(row["text"])
    results.append({"true": row["label"], "pred": pred})
    if (i+1) % 25 == 0:
        acc = sum(1 for r in results if r['true']==r['pred']) / len(results)
        print(f"  {i+1}/{len(val_rows)} — accuracy: {acc:.3f}")

# Compute metrics
tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
for r in results:
    if r['true'] == r['pred']: tp[r['true']] += 1
    else: fp[r['pred']] += 1; fn[r['true']] += 1

print("\n" + "="*60)
for lbl in LABELS:
    sup = sum(1 for r in results if r['true']==lbl)
    p = tp[lbl]/(tp[lbl]+fp[lbl]) if (tp[lbl]+fp[lbl])>0 else 0
    r = tp[lbl]/(tp[lbl]+fn[lbl]) if (tp[lbl]+fn[lbl])>0 else 0
    f = 2*p*r/(p+r) if (p+r)>0 else 0
    print(f"  {lbl:35s} P={p:.4f} R={r:.4f} F1={f:.4f} n={sup}")

correct = sum(1 for r in results if r['true']==r['pred'])
macro_f1 = sum(2*tp[l]/(2*tp[l]+fp[l]+fn[l]) if (2*tp[l]+fp[l]+fn[l])>0 else 0 for l in LABELS)/4
print(f"\nMacro F1: {macro_f1:.4f}  Accuracy: {correct}/{len(results)}")

output = {"model": _CANDIDATE_ID, "quantization": "4bit-qlora", "holdout": len(results),
          "macro_f1": round(macro_f1,4), "accuracy": round(correct/len(results),4), "predictions": results}
Path("/content/eval-results-qlora.json").write_text(json.dumps(output, indent=2), encoding="utf-8")
print("\nSaved to /content/eval-results-qlora.json")

## Step 7: Download artifacts

Download from Colab Files panel:
1. `training-summary.json` — verify it says `quantization_mode: 4bit-qlora`
2. `eval-results-qlora.json` — new F1 numbers
3. The GGUF file (if produced)

Then update the report with the real QLoRA numbers.

In [ ]:
import shutil

# Zip adapter for download
zip_path = Path("/content/adapter-qlora-final.zip")
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', str(adapter_dir.parent), adapter_dir.name)
print(f"Adapter: {zip_path} ({zip_path.stat().st_size // 1024} KB)")

# Copy training summary for easy download
shutil.copy2(adapter_dir / "training-summary.json", "/content/training-summary.json")
print("Training summary: /content/training-summary.json")

# List GGUF files
for g in MODEL_ROOT.rglob("*.gguf"):
    print(f"GGUF: {g} ({g.stat().st_size // (1024*1024)} MB)")